<a href="https://colab.research.google.com/github/RadhikaDeshpande1010/PySpark-RDD-Analytics-Telecom-Usage-Dataset/blob/main/PySpark_RDD_Telecom_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📡 PySpark RDD Analytics — Telecom Usage Dataset

**Domain:** Telecom Analytics | **Engine:** Apache Spark (RDD API) | **Language:** Python 3

---

## 📋 Dataset Schema

| Field | Index | Type | Description |
|-------------|-------|--------|-------------------------------------|
| customer_id | 0 | int | Unique customer identifier |
| name | 1 | str | Customer name |
| city | 2 | str | City of the customer |
| usage_type | 3 | str | Type of usage: Call / Data / SMS |
| amount | 4 | int | Usage amount in units |
| date | 5 | str | Transaction date (YYYY-MM-DD) |

**Sample record:** `1001,John,Delhi,Call,300,2024-01-01`

---

## 📂 Exercises Overview

| Range | Topics Covered |
|------------|--------------------------------------------------|
| Q1 – Q10 | map, filter, reduceByKey, count, distinct |
| Q11 – Q20 | sortByKey, sortBy, aggregations per date/city |
| Q21 – Q30 | Composite keys, min/max per type, top-N |
| Q31 – Q46 | flatMap, chained transforms, rank & slice |

## ⚙️ Environment Setup

Initialize a local SparkSession and retrieve the SparkContext.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local") \
    .appName("Telecom_RDD_Analytics") \
    .getOrCreate()

print(spark)

In [2]:
sc = spark.sparkContext
print(sc)

<SparkContext master=local appName=Telecom_RDD_Analytics>


## 📥 Data Ingestion

Load the raw telecom text file and parse each record into a list of fields.

In [3]:
# Load raw text file into an RDD
raw_rdd = sc.textFile("/content/sample_data/telecom_rdd_data.txt")
print(f"Total raw lines: {raw_rdd.count()}")

Total raw lines: 20


In [4]:
# Parse each line by splitting on comma → list of fields
telecom_rdd = raw_rdd.map(lambda row: row.split(","))
telecom_rdd.take(5)

[['1001', 'John', 'Delhi', 'Call', '300', '2024-01-01'],
 ['1002', 'Amit', 'Mumbai', 'Data', '500', '2024-01-01'],
 ['1003', 'Riya', 'Delhi', 'SMS', '50', '2024-01-01'],
 ['1004', 'Neha', 'Chennai', 'Call', '200', '2024-01-02'],
 ['1005', 'Rahul', 'Mumbai', 'Data', '700', '2024-01-02']]

---
## 🔄 Core Transformations & Aggregations

Exercises 1–10: `map`, `filter`, `distinct`, `reduceByKey`, `count`, `sortByKey`, `sortBy`.

In [5]:
# Q1 — Extract distinct (city, amount) pairs from the dataset
city_amount_pairs = telecom_rdd.map(lambda row: (row[2], int(row[4]))).distinct()
print(city_amount_pairs.collect())

[('Delhi', 300), ('Mumbai', 500), ('Delhi', 50), ('Chennai', 200), ('Mumbai', 700), ('Delhi', 150), ('Kolkata', 40), ('Chennai', 650), ('Delhi', 400), ('Mumbai', 350), ('Kolkata', 800), ('Delhi', 60), ('Chennai', 220), ('Mumbai', 900), ('Delhi', 330), ('Kolkata', 55), ('Chennai', 720), ('Delhi', 410), ('Mumbai', 390), ('Kolkata', 820)]


In [6]:
# Q2 — Count total number of records in the dataset
total_records = telecom_rdd.count()
print(f"Total records: {total_records}")

Total records: 20


In [7]:
# Q3 — Filter all records where usage_type is 'Call'
call_usage_records = telecom_rdd.filter(lambda row: row[3].strip().lower() == "call")
print(call_usage_records.collect())

[['1001', 'John', 'Delhi', 'Call', '300', '2024-01-01'], ['1004', 'Neha', 'Chennai', 'Call', '200', '2024-01-02'], ['1006', 'Sneha', 'Delhi', 'Call', '150', '2024-01-02'], ['1009', 'Pooja', 'Delhi', 'Call', '400', '2024-01-03'], ['1010', 'Arjun', 'Mumbai', 'Call', '350', '2024-01-04'], ['1013', 'Rohit', 'Chennai', 'Call', '220', '2024-01-05'], ['1015', 'Deepak', 'Delhi', 'Call', '330', '2024-01-05'], ['1018', 'Kavita', 'Delhi', 'Call', '410', '2024-01-06'], ['1019', 'Sanjay', 'Mumbai', 'Call', '390', '2024-01-07']]


In [8]:
# Q4 — Total usage amount per city using reduceByKey
total_usage_per_city = telecom_rdd \
    .map(lambda row: (row[2], int(row[4]))) \
    .reduceByKey(lambda x, y: x + y)
print(total_usage_per_city.collect())

[('Delhi', 1700), ('Mumbai', 2840), ('Chennai', 1790), ('Kolkata', 1715)]


In [9]:
# Q5 — Total Call usage amount per city
call_usage_per_city = telecom_rdd \
    .filter(lambda row: row[3].strip().lower() == "call") \
    .map(lambda row: (row[2], int(row[4]))) \
    .reduceByKey(lambda x, y: x + y)
print(call_usage_per_city.collect())

[('Delhi', 1590), ('Chennai', 420), ('Mumbai', 740)]


In [10]:
# Q6 — Total usage per usage_type (Call, Data, SMS)
total_usage_per_type = telecom_rdd \
    .map(lambda row: (row[3], int(row[4]))) \
    .reduceByKey(lambda x, y: x + y)
print(total_usage_per_type.collect())

[('Call', 2750), ('Data', 5090), ('SMS', 205)]


In [11]:
# Q7 — Maximum usage amount per city using reduceByKey
city_max_usage = telecom_rdd \
    .map(lambda row: (row[2], int(row[4]))) \
    .reduceByKey(lambda x, y: max(x, y))
print(city_max_usage.collect())

[('Delhi', 410), ('Mumbai', 900), ('Chennai', 720), ('Kolkata', 820)]


In [12]:
# Q8 — Count number of transactions per city using reduceByKey
transactions_per_city = telecom_rdd \
    .map(lambda row: (row[2], 1)) \
    .reduceByKey(lambda x, y: x + y)
print(transactions_per_city.collect())

[('Delhi', 7), ('Mumbai', 5), ('Chennai', 4), ('Kolkata', 4)]


In [13]:
# Q9 — Sort total usage amount by city name (alphabetical order)
city_usage_sorted_by_name = telecom_rdd \
    .map(lambda row: (row[2], int(row[4]))) \
    .reduceByKey(lambda x, y: x + y) \
    .sortByKey()
print(city_usage_sorted_by_name.collect())

[('Chennai', 1790), ('Delhi', 1700), ('Kolkata', 1715), ('Mumbai', 2840)]


In [14]:
# Q10 — Sort all usage records by date (ascending)
records_sorted_by_date = telecom_rdd.sortBy(lambda row: row[5])
print(records_sorted_by_date.collect())

[['1001', 'John', 'Delhi', 'Call', '300', '2024-01-01'], ['1002', 'Amit', 'Mumbai', 'Data', '500', '2024-01-01'], ['1003', 'Riya', 'Delhi', 'SMS', '50', '2024-01-01'], ['1004', 'Neha', 'Chennai', 'Call', '200', '2024-01-02'], ['1005', 'Rahul', 'Mumbai', 'Data', '700', '2024-01-02'], ['1006', 'Sneha', 'Delhi', 'Call', '150', '2024-01-02'], ['1007', 'Vikas', 'Kolkata', 'SMS', '40', '2024-01-03'], ['1008', 'Ankit', 'Chennai', 'Data', '650', '2024-01-03'], ['1009', 'Pooja', 'Delhi', 'Call', '400', '2024-01-03'], ['1010', 'Arjun', 'Mumbai', 'Call', '350', '2024-01-04'], ['1011', 'Kiran', 'Kolkata', 'Data', '800', '2024-01-04'], ['1012', 'Meena', 'Delhi', 'SMS', '60', '2024-01-04'], ['1013', 'Rohit', 'Chennai', 'Call', '220', '2024-01-05'], ['1014', 'Simran', 'Mumbai', 'Data', '900', '2024-01-05'], ['1015', 'Deepak', 'Delhi', 'Call', '330', '2024-01-05'], ['1016', 'Nisha', 'Kolkata', 'SMS', '55', '2024-01-06'], ['1017', 'Manoj', 'Chennai', 'Data', '720', '2024-01-06'], ['1018', 'Kavita', 'De

---
## 📊 Sorting, Revenue & Composite Key Analysis

Exercises 11–30: descending sorts, date-level aggregations, composite `(city, usage_type)` keys.

In [15]:
# Q11 — Sort cities by total usage amount (descending)
city_total_usage_desc = telecom_rdd \
    .map(lambda row: (row[2], int(row[4]))) \
    .reduceByKey(lambda x, y: x + y) \
    .sortBy(lambda x: x[1], ascending=False)
print(city_total_usage_desc.collect())

[('Mumbai', 2840), ('Chennai', 1790), ('Kolkata', 1715), ('Delhi', 1700)]


In [16]:
# Q12a — Filter all records where amount > 500
high_amount_records = telecom_rdd.filter(lambda row: int(row[4]) > 500)
print(high_amount_records.collect())

[['1005', 'Rahul', 'Mumbai', 'Data', '700', '2024-01-02'], ['1008', 'Ankit', 'Chennai', 'Data', '650', '2024-01-03'], ['1011', 'Kiran', 'Kolkata', 'Data', '800', '2024-01-04'], ['1014', 'Simran', 'Mumbai', 'Data', '900', '2024-01-05'], ['1017', 'Manoj', 'Chennai', 'Data', '720', '2024-01-06'], ['1020', 'Alok', 'Kolkata', 'Data', '820', '2024-01-07']]


In [17]:
# Q12b — Extract all records where usage_type is 'Data' and count them
data_usage_count = telecom_rdd \
    .filter(lambda row: row[3].strip().lower() == "data") \
    .count()
print(f"Data usage record count: {data_usage_count}")

Data usage record count: 7


In [18]:
# Q13 — Find total usage amount per city using reduceByKey
total_usage_per_city = telecom_rdd \
    .map(lambda row: (row[2], int(row[4]))) \
    .reduceByKey(lambda x, y: x + y)
print(total_usage_per_city.collect())

[('Delhi', 1700), ('Mumbai', 2840), ('Chennai', 1790), ('Kolkata', 1715)]


In [19]:
# Q14 — Calculate total Call usage amount across all cities
total_call_usage_all_cities = telecom_rdd \
    .filter(lambda row: row[3].strip().lower() == "call") \
    .map(lambda row: ("total", int(row[4]))) \
    .reduceByKey(lambda x, y: x + y)
print(total_call_usage_all_cities.collect())

[('total', 2750)]


In [20]:
# Q15 — Find total usage amount per date using reduceByKey
total_usage_per_date = telecom_rdd \
    .map(lambda row: (row[5], int(row[4]))) \
    .reduceByKey(lambda x, y: x + y)
print(total_usage_per_date.collect())

[('2024-01-01', 850), ('2024-01-02', 1050), ('2024-01-03', 1090), ('2024-01-04', 1210), ('2024-01-05', 1450), ('2024-01-06', 1185), ('2024-01-07', 1210)]


In [21]:
# Q16 — Key by usage_type and calculate total usage for each type
total_usage_by_type = telecom_rdd \
    .map(lambda row: (row[3], int(row[4]))) \
    .reduceByKey(lambda x, y: x + y)
print(total_usage_by_type.collect())

[('Call', 2750), ('Data', 5090), ('SMS', 205)]


In [22]:
# Q17 — Count number of transactions per city using reduceByKey
transactions_per_city = telecom_rdd \
    .map(lambda row: (row[2], 1)) \
    .reduceByKey(lambda x, y: x + y)
print(transactions_per_city.collect())

[('Delhi', 7), ('Mumbai', 5), ('Chennai', 4), ('Kolkata', 4)]


In [23]:
# Q18 — Total usage per city, sorted by city name using sortByKey
city_usage_sorted = telecom_rdd \
    .map(lambda row: (row[2], int(row[4]))) \
    .reduceByKey(lambda x, y: x + y) \
    .sortByKey()
print(city_usage_sorted.collect())

[('Chennai', 1790), ('Delhi', 1700), ('Kolkata', 1715), ('Mumbai', 2840)]


In [24]:
# Q19 — Sort total usage per date by usage amount (descending)
usage_per_date_sorted_desc = telecom_rdd \
    .map(lambda row: (row[5], int(row[4]))) \
    .reduceByKey(lambda x, y: x + y) \
    .sortBy(lambda x: x[1], ascending=False)
print(usage_per_date_sorted_desc.collect())

[('2024-01-05', 1450), ('2024-01-04', 1210), ('2024-01-07', 1210), ('2024-01-06', 1185), ('2024-01-03', 1090), ('2024-01-02', 1050), ('2024-01-01', 850)]


In [25]:
# Q20 — Find the city with maximum total usage (sort ascending, last entry is max)
city_max_total_usage = telecom_rdd \
    .map(lambda row: (row[2], int(row[4]))) \
    .reduceByKey(lambda x, y: x + y) \
    .sortBy(lambda x: x[1], ascending=False)
print(city_max_total_usage.first())

('Mumbai', 2840)


---
## 🔍 Advanced Filters, flatMap & Top-N Queries

Exercises 21–46: composite key queries, `flatMap`, chained transforms, and slice operations.

In [26]:
# Q21 — Filter all transactions where city is 'Delhi' and amount > 300
delhi_high_value = telecom_rdd.filter(
    lambda row: row[2].strip().lower() == "delhi" and int(row[4]) > 300
)
print(delhi_high_value.collect())

[['1009', 'Pooja', 'Delhi', 'Call', '400', '2024-01-03'], ['1015', 'Deepak', 'Delhi', 'Call', '330', '2024-01-05'], ['1018', 'Kavita', 'Delhi', 'Call', '410', '2024-01-06']]


In [27]:
# Q22 — Extract SMS records and calculate total SMS usage amount
sms_total_usage = telecom_rdd \
    .filter(lambda row: row[3].strip().lower() == "sms") \
    .map(lambda row: (row[3], int(row[4]))) \
    .reduceByKey(lambda x, y: x + y)
print(sms_total_usage.collect())

[('SMS', 205)]


In [28]:
# Q23 — Find minimum usage amount per city using reduceByKey
city_min_usage = telecom_rdd \
    .map(lambda row: (row[2], int(row[4]))) \
    .reduceByKey(lambda x, y: min(x, y))
print(city_min_usage.collect())

[('Delhi', 50), ('Mumbai', 350), ('Chennai', 200), ('Kolkata', 40)]


In [29]:
# Q24 — Find maximum usage amount per usage_type using reduceByKey
type_max_usage = telecom_rdd \
    .map(lambda row: (row[3], int(row[4]))) \
    .reduceByKey(lambda x, y: max(x, y))
print(type_max_usage.collect())

[('Call', 410), ('Data', 900), ('SMS', 60)]


In [30]:
# Q25 — Calculate total usage per (city, date) composite key
city_date_usage = telecom_rdd \
    .map(lambda row: ((row[2], row[5]), int(row[4]))) \
    .reduceByKey(lambda x, y: x + y)
print(city_date_usage.collect())

[(('Delhi', '2024-01-01'), 350), (('Mumbai', '2024-01-01'), 500), (('Chennai', '2024-01-02'), 200), (('Mumbai', '2024-01-02'), 700), (('Delhi', '2024-01-02'), 150), (('Kolkata', '2024-01-03'), 40), (('Chennai', '2024-01-03'), 650), (('Delhi', '2024-01-03'), 400), (('Mumbai', '2024-01-04'), 350), (('Kolkata', '2024-01-04'), 800), (('Delhi', '2024-01-04'), 60), (('Chennai', '2024-01-05'), 220), (('Mumbai', '2024-01-05'), 900), (('Delhi', '2024-01-05'), 330), (('Kolkata', '2024-01-06'), 55), (('Chennai', '2024-01-06'), 720), (('Delhi', '2024-01-06'), 410), (('Mumbai', '2024-01-07'), 390), (('Kolkata', '2024-01-07'), 820)]


In [31]:
# Q26 — Count transactions per (city, usage_type) composite key
city_type_count = telecom_rdd \
    .map(lambda row: ((row[2], row[3]), 1)) \
    .reduceByKey(lambda x, y: x + y)
print(city_type_count.collect())

[(('Delhi', 'Call'), 5), (('Mumbai', 'Data'), 3), (('Delhi', 'SMS'), 2), (('Chennai', 'Call'), 2), (('Kolkata', 'SMS'), 2), (('Chennai', 'Data'), 2), (('Mumbai', 'Call'), 2), (('Kolkata', 'Data'), 2)]


In [32]:
# Q27 — Total Data usage per city for records where amount > 600
high_data_usage_per_city = telecom_rdd \
    .filter(lambda row: row[3].strip().lower() == "data" and int(row[4]) > 600) \
    .map(lambda row: (row[2], int(row[4]))) \
    .reduceByKey(lambda x, y: x + y)
print(high_data_usage_per_city.collect())

[('Mumbai', 1600), ('Chennai', 1370), ('Kolkata', 1620)]


In [33]:
# Q28 — Top 5 highest usage transactions using sortByKey (no groupByKey)
top_five_transactions = telecom_rdd \
    .map(lambda row: (int(row[4]), row)) \
    .sortByKey(ascending=False) \
    .take(5)
print(top_five_transactions)

[(900, ['1014', 'Simran', 'Mumbai', 'Data', '900', '2024-01-05']), (820, ['1020', 'Alok', 'Kolkata', 'Data', '820', '2024-01-07']), (800, ['1011', 'Kiran', 'Kolkata', 'Data', '800', '2024-01-04']), (720, ['1017', 'Manoj', 'Chennai', 'Data', '720', '2024-01-06']), (700, ['1005', 'Rahul', 'Mumbai', 'Data', '700', '2024-01-02'])]


In [34]:
# Q29 — Sort usage types by total usage amount (descending)
usage_type_ranked = telecom_rdd \
    .map(lambda row: (row[3], int(row[4]))) \
    .reduceByKey(lambda x, y: x + y) \
    .sortBy(lambda x: x[1], ascending=False)
print(usage_type_ranked.collect())

[('Data', 5090), ('Call', 2750), ('SMS', 205)]


In [35]:
# Q30 — Find the date with lowest total revenue using reduceByKey and sortBy
lowest_revenue_date = telecom_rdd \
    .map(lambda row: (row[5], int(row[4]))) \
    .reduceByKey(lambda x, y: x + y) \
    .sortBy(lambda x: x[1], ascending=True)
print(lowest_revenue_date.first())

('2024-01-01', 850)


In [36]:
# Q31 — Extract (city, amount) pairs and filter records where amount > 400
city_high_amount = telecom_rdd \
    .map(lambda row: (row[2], int(row[4]))) \
    .filter(lambda x: x[1] > 400)
print(city_high_amount.collect())

[('Mumbai', 500), ('Mumbai', 700), ('Chennai', 650), ('Kolkata', 800), ('Mumbai', 900), ('Chennai', 720), ('Delhi', 410), ('Kolkata', 820)]


In [37]:
# Q32 — Extract (usage_type, amount) pairs and filter distinct 'Call' records
call_type_usage = telecom_rdd \
    .map(lambda row: (row[3], int(row[4]))) \
    .filter(lambda x: x[0].strip().lower() == "call") \
    .distinct()
print(call_type_usage.collect())

[('Call', 300), ('Call', 200), ('Call', 150), ('Call', 400), ('Call', 350), ('Call', 220), ('Call', 330), ('Call', 410), ('Call', 390)]


In [38]:
# Q33 — Normalize city names to lowercase, count records where city == 'delhi'
delhi_record_count = telecom_rdd \
    .map(lambda row: row[2].strip().lower()) \
    .filter(lambda city: city == "delhi") \
    .count()
print(f"Delhi record count: {delhi_record_count}")

Delhi record count: 7


In [39]:
# Q34 — Use flatMap to split each record into individual fields, count all elements
total_elements = telecom_rdd.flatMap(lambda row: row)
print(f"Total individual elements: {total_elements.count()}")

Total individual elements: 120


In [40]:
# Q35 — Extract all usage types using flatMap and remove duplicates
distinct_usage_types = telecom_rdd \
    .flatMap(lambda row: [row[3].strip()]) \
    .distinct()
print(distinct_usage_types.collect())

['Call', 'Data', 'SMS']


In [41]:
# Q36 — Total usage amount per city using reduceByKey
city_total_usage = telecom_rdd \
    .map(lambda row: (row[2], int(row[4]))) \
    .reduceByKey(lambda x, y: x + y)
print(city_total_usage.collect())

[('Delhi', 1700), ('Mumbai', 2840), ('Chennai', 1790), ('Kolkata', 1715)]


In [42]:
# Q37 — Find grand total usage amount across all cities
# Step 1: compute total per city
# Step 2: extract city totals only
# Step 3: sum all city totals via reduce
grand_total_usage = telecom_rdd \
    .map(lambda row: (row[2], int(row[4]))) \
    .reduceByKey(lambda x, y: x + y) \
    .map(lambda x: x[1]) \
    .reduce(lambda x, y: x + y)
print(f"Grand total usage across all cities: {grand_total_usage}")

Grand total usage across all cities: 8045


In [43]:
# Q38 — Calculate total usage per usage_type
total_usage_per_type = telecom_rdd \
    .map(lambda row: (row[3], int(row[4]))) \
    .reduceByKey(lambda x, y: x + y)
print(total_usage_per_type.collect())

[('Call', 2750), ('Data', 5090), ('SMS', 205)]


In [44]:
# Q39 — Find maximum usage amount per city using reduceByKey
city_max_usage = telecom_rdd \
    .map(lambda row: (row[2], int(row[4]))) \
    .reduceByKey(lambda x, y: max(x, y))
print(city_max_usage.collect())

[('Delhi', 410), ('Mumbai', 900), ('Chennai', 720), ('Kolkata', 820)]


In [45]:
# Q40 — Count number of transactions per date
transactions_per_date = telecom_rdd \
    .map(lambda row: (row[5], 1)) \
    .reduceByKey(lambda x, y: x + y)
print(transactions_per_date.collect())

[('2024-01-01', 3), ('2024-01-02', 3), ('2024-01-03', 3), ('2024-01-04', 3), ('2024-01-05', 3), ('2024-01-06', 3), ('2024-01-07', 2)]


In [46]:
# Q41 — Total usage per city, sorted by city name (ascending)
city_usage_sorted_asc = telecom_rdd \
    .map(lambda row: (row[2], int(row[4]))) \
    .reduceByKey(lambda x, y: x + y) \
    .sortBy(lambda x: x[0], ascending=True)
print(city_usage_sorted_asc.collect())

[('Chennai', 1790), ('Delhi', 1700), ('Kolkata', 1715), ('Mumbai', 2840)]


In [47]:
# Q42 — Sort total usage per date in ascending date order
usage_per_date_sorted_asc = telecom_rdd \
    .map(lambda row: (row[5], int(row[4]))) \
    .reduceByKey(lambda x, y: x + y) \
    .sortBy(lambda x: x[0])
print(usage_per_date_sorted_asc.collect())

[('2024-01-01', 850), ('2024-01-02', 1050), ('2024-01-03', 1090), ('2024-01-04', 1210), ('2024-01-05', 1450), ('2024-01-06', 1185), ('2024-01-07', 1210)]


In [48]:
# Q43 — Find top 5 highest transaction amounts using sortBy
top_five_amounts = telecom_rdd \
    .map(lambda row: int(row[4])) \
    .sortBy(lambda x: x, ascending=False) \
    .take(5)
print(top_five_amounts)

[900, 820, 800, 720, 700]


In [49]:
# Q45 — Find lowest 3 transactions by amount
lowest_three_amounts = telecom_rdd \
    .map(lambda row: int(row[4])) \
    .sortBy(lambda x: x, ascending=True) \
    .take(3)
print(lowest_three_amounts)

[40, 50, 55]


In [50]:
# Q46 — Sort (city, total_usage) pairs by city name (descending)
city_usage_sorted_desc = telecom_rdd \
    .map(lambda row: (row[2], int(row[4]))) \
    .reduceByKey(lambda x, y: x + y) \
    .sortBy(lambda x: x[0], ascending=False)
print(city_usage_sorted_desc.collect())

[('Mumbai', 2840), ('Kolkata', 1715), ('Delhi', 1700), ('Chennai', 1790)]
